# Key enrollment and rotation

Install Deno and register its TypeScript kernel with `deno jupyter --install`. Open this file
in a Jupyter front end such as JupyterLab or VS Code, select **Deno**, then Run All in a fresh kernel.
The cells import the exact published verifier **2.0.0**.

Verification returns facts; the host still owns trusted inputs, authorization, replay and revocation.
Only safe semantic results are displayed; generated keys, signatures and nonces stay in memory.


## Load the published verifier

Check the package identity and the portable backend Deno selects.


In [ ]:
import { Buffer } from "node:buffer";
import { BACKEND_ID, acceptablePublicKey, torsionFree, assembleCompact, grantSigningInput, jwkFromPublicKey, thumbprint, untrustedKeyLocator, verifyGrant, type GrantFacts, type Result, type SigningInput, type TrustedIssuer } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import { generateKeyPairSync, sign, type KeyObject } from "node:crypto";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
import assert from "node:assert/strict";
assert.equal(verifierPackage.version, "2.0.0");
assert.equal(BACKEND_ID, "portable");
console.log(JSON.stringify({ version: verifierPackage.version, backend: BACKEND_ID }));


{"version":"2.0.0","backend":"portable"}


## Prepare enrolled issuer keys

Keys enter the trusted map through host enrollment. The untrusted header is only a lookup hint.


In [ ]:
const publicKeyBytes = (key: KeyObject): Uint8Array => Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const signingBytes = (input: SigningInput): Uint8Array => Buffer.concat([input.protectedSegment, Buffer.from("."), input.payloadSegment]);
const holder = publicKeyBytes(generateKeyPairSync("ed25519").publicKey);

function issue(keyId: string, privateKey: KeyObject, grantId: string): Uint8Array {
  const input = grantSigningInput({
    keyId, issuer: "https://issuer.example", grantId, audiences: ["https://storage.example"],
    issuedAt: 1_000, notBefore: 1_000, expiresAt: 100_000,
    holderThumbprint: thumbprint(jwkFromPublicKey(holder)),
    operations: [{ name: "objects.list", selectors: ["all"] }],
  });
  assert.ok(input.ok);
  const compact = assembleCompact(input.value, sign(null, signingBytes(input.value), privateKey));
  assert.ok(compact.ok);
  return compact.value;
}

const oldKey = generateKeyPairSync("ed25519");
const newKey = generateKeyPairSync("ed25519");
const trusted = new Map<string, TrustedIssuer>([
  ["issuer-2026-09", { keyId: "issuer-2026-09", publicKey: publicKeyBytes(oldKey.publicKey) }],
  ["issuer-2026-10", { keyId: "issuer-2026-10", publicKey: publicKeyBytes(newKey.publicKey) }],
]);

function verifyAny(grant: Uint8Array): Result<GrantFacts> {
  const locator = untrustedKeyLocator(grant);
  const key = locator.ok ? trusted.get(locator.value.keyId) : undefined;
  if (key === undefined) return { ok: false };
  return verifyGrant(grant, key, {
    issuer: "https://issuer.example", audience: "https://storage.example", evaluationTime: 2_000, clockSkew: 60,
  });
}


console.log(JSON.stringify({ enrolledKeys: trusted.size }));


{"enrolledKeys":2}


## Check enrollment properties

Reject unacceptable Ed25519 keys before enrollment.


In [ ]:
const acceptable = acceptablePublicKey(publicKeyBytes(newKey.publicKey));
const primeOrder = torsionFree(publicKeyBytes(newKey.publicKey));
const zeroKey = torsionFree(new Uint8Array(32));
assert.equal(acceptable, true);
assert.equal(primeOrder, true);
assert.equal(zeroKey, false);
console.log(JSON.stringify({ acceptable, primeOrder, zeroKey }));


{"acceptable":true,"primeOrder":true,"zeroKey":false}


## Verify during overlap

Both enrolled keys can verify grants during the rotation window.


In [ ]:
const before = issue("issuer-2026-09", oldKey.privateKey, "grant-a");
const after = issue("issuer-2026-10", newKey.privateKey, "grant-b");
const oldResult = verifyAny(before);
const newResult = verifyAny(after);
assert.ok(oldResult.ok);
assert.ok(newResult.ok);
console.log(JSON.stringify({ old: oldResult.ok, next: newResult.ok }));


{"old":true,"next":true}


## Refuse the wrong signer

The key identifier cannot override signature verification.


In [ ]:
const wrongSigner = verifyAny(issue("issuer-2026-10", oldKey.privateKey, "grant-c"));
assert.deepEqual(wrongSigner, { ok: false });
console.log(JSON.stringify(wrongSigner));


{"ok":false}


## Retire the old enrollment

Once old grants have expired operationally, remove the old enrollment and confirm that only the new key remains usable.


In [ ]:
trusted.delete("issuer-2026-09");
const retired = verifyAny(before);
const retained = verifyAny(after);
assert.deepEqual(retired, { ok: false });
assert.ok(retained.ok);
console.log(JSON.stringify({ retired: retired.ok, retained: retained.ok, enrolledKeys: trusted.size }));


{"retired":false,"retained":true,"enrolledKeys":1}


## Finish

No private material was written to disk. Shut down the kernel to discard the demonstration keys.
Continue with the [getting started guide](../guides/getting-started.md).
